In [8]:
%pip install chromadb 

%pip install faiss-cpu 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
%pip install torch sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


C:\Users\Siddhanth\AppData\Local\Temp\ipykernel_4012\1080596676.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] ='pdf'
            
            all_documents.extend(documents)
            print(f"loaded{len(documents)} pages")
        except:
            print(f"Error:{e}")
    print(f"total documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_documents = process_all_pdfs("../data")
            

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)


Found 3 PDF files to process

Processing: Axxela.pdf
loaded1 pages

Processing: UBS.pdf
loaded3 pages

Processing: ZS_Case_Frameworks.pdf
loaded16 pages
total documents loaded: 20


In [ ]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """spit documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size, 
        chunk_overlap = chunk_overlap,
        length_function = len, 
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents  into {len(split_docs)} chunks")

    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [ ]:
chunks = split_documents(all_pdf_documents)

chunks

Split 20 documents  into 31 chunks

Example chunk:
Content: About Us: 
We are a firm with a strong presence in the Futures and Derivatives related operations in the 
International Capital Markets. We have built ourselves a reputation for grooming university 
g
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-03T13:17:45+00:00', 'author': 'Megha Gupta', 'moddate': '2026-06-03T13:17:46+00:00', 'source': '..\\data\\pdf\\Axxela.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Axxela.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-06-03T13:17:45+00:00', 'author': 'Megha Gupta', 'moddate': '2026-06-03T13:17:46+00:00', 'source': '..\\data\\pdf\\Axxela.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'Axxela.pdf', 'file_type': 'pdf'}, page_content='About Us: \nWe are a firm with a strong presence in the Futures and Derivatives related operations in the \nInternational Capital Markets. We have built ourselves a reputation for grooming university \ngraduates, aspiring to leverage their analytical skills and emotional intelligence, into quick \nthinker’s adept at handling competitive situations professionally. \nWith this email we want to offer immense growth opportunities to your students. For further \ndetails: - https://www.axxela.in/ \n \nRole Details: \n⮚ To be involved in the execution of trades in commodities and fixed income in International \nMarkets. \n⮚ Utilize strong risk managem

In [10]:


import numpy as np 
from sentence_transformers import SentenceTransformer 
import chromadb 
from chromadb.config import Settings 
import uuid 
from typing  import List, Dict, Any, Tuple 
from sklearn.metrics.pairwise  import cosine_similarity

C:\Users\Siddhanth\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class EmbeddedManager:
    """Handles document embeddings  generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialise the embedding manager
        
        Args:"""